# Manga Panel Assistant — Colab runner

This notebook holds **no logic**. It clones the repo, installs it, and calls the modules.
All code lives in `src/manga/`, all parameters in `configs/`.

**Runtime → Change runtime type → T4 GPU → Save.**

After a disconnect: run cells 1 and 2, then carry on.

## 1 — clone, install, mount Drive

`HF_HOME` points at Drive so the ~8 GB of model weights survive a disconnect.

In [ ]:
REPO = 'https://github.com/<you>/manga-assistant.git'  # <- edit

import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/manga_poc'
os.environ['HF_HOME'] = f'{DRIVE}/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

![ -d manga-assistant ] || git clone -q $REPO
%cd /content/manga-assistant
!git pull -q
!pip install -q -e .

!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 — load models

First run downloads ~8 GB to Drive. Later runs load from Drive in a minute or two.

In [ ]:
from manga import generate, pipeline, prompts, sketches
from manga.config import load
from manga.utils import grid, save

cfg = load('conan.yaml')
models = pipeline.load(cfg)

print('cn_scale', cfg.generation.cn_scale,
      '| cn_end', cfg.generation.cn_end,
      '| ip_scale', cfg.generation.ip_scale)

### point the data paths at Drive

Keep references and outlines on Drive rather than inside the repo — they are
DVC's job, and the repo should stay small.

In [ ]:
cfg['paths']['refs']     = f'{DRIVE}/refs'
cfg['paths']['outlines'] = f'{DRIVE}/outlines'
cfg['paths']['results']  = f'{DRIVE}/results'
cfg['character']['reference'] = f"{DRIVE}/refs/conan_ref_clean.png"

## 3 — prepare a reference

IP-Adapter copies **everything** in the reference: background, props, ink splatter.
Whiten the background before using it.

`denoise=True` helps on grainy renders but **smears screentone** — leave it off for
real manga art.

In [ ]:
from PIL import Image

raw = Image.open(f'{DRIVE}/refs/conan_ref.png')
clean = sketches.whiten_background(raw, threshold=90, denoise=True)
clean.save(f'{DRIVE}/refs/conan_ref_clean.png')
grid([raw, clean], cols=2)

## 4 — one pose

Use `load_photo_sketch` for a photograph of a paper sketch, `load_sketch` for a
clean digital one.

In [ ]:
pose = cfg.poses[1]                      # walking, from the side
control = sketches.load_sketch(f"{cfg.paths.outlines}/{pose['file']}")
reference = sketches.load_reference(cfg.character.reference)

images = generate.outline_to_image(
    models, cfg,
    prompts.character_prompt(cfg, pose['text']),
    prompts.character_negative(cfg),
    control, reference, n=2,
)
grid([control] + images, cols=3)

## 5 — every pose

The evidence grid. Sketch in the first column, two attempts beside it.

In [ ]:
tiles = []
for pose in cfg.poses:
    control = sketches.load_sketch(f"{cfg.paths.outlines}/{pose['file']}")
    tiles += [control] + generate.outline_to_image(
        models, cfg,
        prompts.character_prompt(cfg, pose['text']),
        prompts.character_negative(cfg),
        control, reference, n=2,
    )

g = grid(tiles, cols=3)
save(g, f"{cfg.paths.results}/{cfg.character.id}_b3.png")
g

## 6 — ablation sweep

One image per value. `cn_scale` for outline strength, `ip_scale` for identity
strength, `cn_end` for how long ControlNet keeps steering.

In [ ]:
values = [0.4, 0.6, 0.8]
images = generate.sweep(
    models, cfg,
    prompts.character_prompt(cfg, cfg.poses[1]['text']),
    prompts.character_negative(cfg),
    control, reference, 'ip_scale', values,
)
print(values)
grid([control] + images, cols=len(images) + 1)

## 7 — train a per-artist LoRA

Preprocess, tag, train. Roughly 1–2 h on a T4 at the defaults in `configs/conan.yaml`.

In [ ]:
# 1. Manga109 -> single-character crops, speech bubbles filtered out
!python -m manga.preprocess --root $DRIVE/Manga109 --book DetectiveConan \
    --out $DRIVE/datasets/conan --trigger aoymstyle

# 2. real tags instead of the placeholder captions
![ -d sd-scripts ] || git clone -q https://github.com/kohya-ss/sd-scripts
!pip install -q -r sd-scripts/requirements.txt
!bash scripts/tag_dataset.sh $DRIVE/datasets/conan aoymstyle

# 3. train
!python -m manga.train_lora --config conan.yaml --data $DRIVE/datasets/conan

## 8 — B3 vs B4

The headline figure for the report: the same poses, with and without the LoRA.

In [ ]:
pipeline.load_lora(models, f'{DRIVE}/loras/detective_conan/detective_conan_style.safetensors')

tiles = []
for pose in cfg.poses:
    control = sketches.load_sketch(f"{cfg.paths.outlines}/{pose['file']}")
    tiles += [control] + generate.outline_to_image(
        models, cfg,
        prompts.character_prompt(cfg, pose['text']),
        prompts.character_negative(cfg),
        control, reference, n=2,
    )

g = grid(tiles, cols=3)
save(g, f"{cfg.paths.results}/{cfg.character.id}_b4.png")
g